<a href="https://colab.research.google.com/github/nuhuynhh/AAI2026/blob/main/ML_Basics_Part_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Data source: Kaggle - Telco Customer Churn Dataset
# https://www.kaggle.com/datasets/blastchar/telco-customer-churn

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")


In [10]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Segmentation features
df = df[["TotalCharges", "tenure", "SeniorCitizen", "InternetService"]].copy()

# Fix TotalCharges (stored as text)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

df = df.rename(columns={
    "TotalCharges": "annual_spending",
    "tenure": "purchase_frequency",
    "SeniorCitizen": "age",
    "InternetService": "region"
})
# Preprocess data: Select numerical features and scale them

features = ['annual_spending', 'purchase_frequency', 'age']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [14]:
# Determine optimal number of clusters using elbow method
inertia = []
K = range(1, 6)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
# Plot elbow curve
plt.figure(figsize=(8, 5))
plt.plot(list(K), inertia, 'bo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.savefig('elbow_plot.png', dpi=200)
plt.close()

# Apply K-Means with optimal K (e.g., 3 based on elbow method)
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Analyze clusters
cluster_summary = df.groupby('cluster')[features].mean().round(2)
print("Cluster Characteristics:")
print(cluster_summary)

Cluster Characteristics:
         annual_spending  purchase_frequency  age
cluster                                          
0                 753.02               14.88  0.0
1                4350.67               58.64  0.0
2                2810.47               33.30  1.0


In [15]:
# Example of targeted strategies
for cluster in range(optimal_k):
    print(f"\nCluster {cluster} Strategy:")

    if cluster_summary.loc[cluster, 'annual_spending'] > cluster_summary['annual_spending'].median():
        print("High-spending customers: Offer exclusive promotions or loyalty rewards.")
    elif cluster_summary.loc[cluster, 'purchase_frequency'] > cluster_summary['purchase_frequency'].median():
        print("Frequent buyers: Provide bundles, subscription plans, or priority support.")
    else:
        print("Low-engagement customers: Send personalized re-engagement campaigns and limited-time discounts.")



Cluster 0 Strategy:
Low-engagement customers: Send personalized re-engagement campaigns and limited-time discounts.

Cluster 1 Strategy:
High-spending customers: Offer exclusive promotions or loyalty rewards.

Cluster 2 Strategy:
Low-engagement customers: Send personalized re-engagement campaigns and limited-time discounts.


In [16]:
# Save cluster assignments to CSV
df.to_csv('customer_segments.csv', index=False)
print("\nSaved: customer_segments.csv and elbow_plot.png")


Saved: customer_segments.csv and elbow_plot.png
